In [1]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
import pandas as pd

from utils import build_user_vector_document, build_anime_user_vector_document, build_anime_vector_document, find_project_root

In [2]:
project_root = find_project_root()

embeddings = OllamaEmbeddings(model="nomic-embed-text")
animes_users_df = pd.read_csv(project_root / "datasets/mal_2023/1000/animes_users_merged.csv")
users_df = pd.read_csv(project_root / "datasets/mal_2023/1000/users.csv")
animes_df = pd.read_csv(project_root / "datasets/mal_2023/1000/animes.csv")
users_df['username_lower'] = users_df['username'].str.lower()
animes_df['name_lower'] = animes_df['name'].str.lower()
animes_users_df['username_lower'] = animes_users_df['username'].str.lower()


In [3]:
documents = []
ids = []

total_animes = len(animes_df)
print(f"Processando {total_animes} animes...")

for idx, row in animes_df.iterrows():
    if (idx + 1) % 100 == 0 or (idx + 1) == total_animes:
        print(f"Processado: {idx + 1}/{total_animes} ({(idx + 1)/total_animes*100:.1f}%)")
    
    page_content, metadata = build_anime_vector_document(row)
    document = Document(
        page_content=page_content,
        metadata=metadata
    )
    documents.append(document)
    ids.append(str(row['anime_id']))

print(f"\n✅ Criados {len(documents)} documentos de animes. Iniciando inserção no vector store...")

vector_store_anime = Chroma(
    collection_name="animes",
    persist_directory=str(project_root / "datasets/mal_2023/1000/chrome_db/animes"),
    embedding_function=embeddings
)

batch_size = 1000
total_batches = (len(documents) + batch_size - 1) // batch_size

for batch_idx, i in enumerate(range(0, len(documents), batch_size), 1):
    print(f"Inserindo batch {batch_idx}/{total_batches}...")
    batch_documents = documents[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]
    vector_store_anime.add_documents(documents=batch_documents, ids=batch_ids)

print("🚀 Animes concluídos!")

Processando 1461 animes...
Processado: 100/1461 (6.8%)
Processado: 200/1461 (13.7%)
Processado: 300/1461 (20.5%)
Processado: 400/1461 (27.4%)
Processado: 500/1461 (34.2%)
Processado: 600/1461 (41.1%)
Processado: 700/1461 (47.9%)
Processado: 800/1461 (54.8%)
Processado: 900/1461 (61.6%)
Processado: 1000/1461 (68.4%)
Processado: 1100/1461 (75.3%)
Processado: 1200/1461 (82.1%)
Processado: 1300/1461 (89.0%)
Processado: 1400/1461 (95.8%)
Processado: 1461/1461 (100.0%)

✅ Criados 1461 documentos de animes. Iniciando inserção no vector store...
Inserindo batch 1/2...
Inserindo batch 2/2...
🚀 Animes concluídos!


In [4]:
documents = []
ids = []

total_users = len(users_df)
print(f"Processando {total_users} usuários...")

for idx, row in users_df.iterrows():
    if (idx + 1) % 50 == 0 or (idx + 1) == total_users:
        print(f"Processado: {idx + 1}/{total_users} ({(idx + 1)/total_users*100:.1f}%)")
    
    page_content, metadata = build_user_vector_document(row)
    document = Document(
        page_content=page_content,
        metadata=metadata
    )
    
    documents.append(document)
    ids.append(str(row['mal_id']))

print(f"\n✅ Criados {len(documents)} documentos de usuários. Iniciando inserção no vector store...")

vector_store = Chroma(
    collection_name="users",
    persist_directory=str(project_root / "datasets/mal_2023/1000/chrome_db/users"),
    embedding_function=embeddings
)

batch_size = 1000
total_batches = (len(documents) + batch_size - 1) // batch_size

for batch_idx, i in enumerate(range(0, len(documents), batch_size), 1):
    print(f"Inserindo batch {batch_idx}/{total_batches}...")
    batch_documents = documents[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]
    vector_store.add_documents(documents=batch_documents, ids=batch_ids)

print("🚀 Usuários concluídos!")

Processando 1000 usuários...
Processado: 50/1000 (5.0%)
Processado: 100/1000 (10.0%)
Processado: 150/1000 (15.0%)
Processado: 200/1000 (20.0%)
Processado: 250/1000 (25.0%)
Processado: 300/1000 (30.0%)
Processado: 350/1000 (35.0%)
Processado: 400/1000 (40.0%)
Processado: 450/1000 (45.0%)
Processado: 500/1000 (50.0%)
Processado: 550/1000 (55.0%)
Processado: 600/1000 (60.0%)
Processado: 650/1000 (65.0%)
Processado: 700/1000 (70.0%)
Processado: 750/1000 (75.0%)
Processado: 800/1000 (80.0%)
Processado: 850/1000 (85.0%)
Processado: 900/1000 (90.0%)
Processado: 950/1000 (95.0%)
Processado: 1000/1000 (100.0%)

✅ Criados 1000 documentos de usuários. Iniciando inserção no vector store...
Inserindo batch 1/1...
🚀 Usuários concluídos!


In [5]:
documents = []
ids = []

total_rows = len(animes_users_df)
print(f"Processando {total_rows} interações anime-usuário...")

for idx, row in enumerate(animes_users_df.itertuples(), 1):
    if idx % 100 == 0 or idx == total_rows:
        print(f"Processado: {idx}/{total_rows} ({idx/total_rows*100:.1f}%)")
    
    page_content, metadata = build_anime_user_vector_document(row)
    
    # Ensure metadata is a dictionary
    if not isinstance(metadata, dict):
        metadata = {}
        
    # Add the row index to the metadata for identification
    metadata['row_index'] = row.Index
    documents.append(Document(page_content=page_content, metadata=metadata))
    ids.append(f"interaction_{row.Index}")

print(f"\n✅ Criados {len(documents)} documentos. Iniciando inserção no vector store...")

vector_store = Chroma(
    collection_name="animes_users",
    persist_directory=str(project_root / "datasets/mal_2023/1000/chrome_db/animes_users"),
    embedding_function=embeddings
)

batch_size = 1000
total_batches = (len(documents) + batch_size - 1) // batch_size

for batch_idx, i in enumerate(range(0, len(documents), batch_size), 1):
    print(f"Inserindo batch {batch_idx}/{total_batches}...")
    batch_documents = documents[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]
    vector_store.add_documents(documents=batch_documents, ids=batch_ids)

print("🚀 Concluído!")


Processando 250209 interações anime-usuário...
Processado: 100/250209 (0.0%)
Processado: 200/250209 (0.1%)
Processado: 300/250209 (0.1%)
Processado: 400/250209 (0.2%)
Processado: 500/250209 (0.2%)
Processado: 600/250209 (0.2%)
Processado: 700/250209 (0.3%)
Processado: 800/250209 (0.3%)
Processado: 900/250209 (0.4%)
Processado: 1000/250209 (0.4%)
Processado: 1100/250209 (0.4%)
Processado: 1200/250209 (0.5%)
Processado: 1300/250209 (0.5%)
Processado: 1400/250209 (0.6%)
Processado: 1500/250209 (0.6%)
Processado: 1600/250209 (0.6%)
Processado: 1700/250209 (0.7%)
Processado: 1800/250209 (0.7%)
Processado: 1900/250209 (0.8%)
Processado: 2000/250209 (0.8%)
Processado: 2100/250209 (0.8%)
Processado: 2200/250209 (0.9%)
Processado: 2300/250209 (0.9%)
Processado: 2400/250209 (1.0%)
Processado: 2500/250209 (1.0%)
Processado: 2600/250209 (1.0%)
Processado: 2700/250209 (1.1%)
Processado: 2800/250209 (1.1%)
Processado: 2900/250209 (1.2%)
Processado: 3000/250209 (1.2%)
Processado: 3100/250209 (1.2%)
P